# Ejercicio 1 — Detección de curvatura: diseño $3^2$ vs. $2^2$

**Objetivo.** Comparar la capacidad de un diseño factorial a dos niveles ($2^2$) y a
tres niveles ($3^2$) para detectar curvatura en la relación entre los factores y la
respuesta. Ajustar modelos lineal y cuadrático, diagnosticar falta de ajuste y concluir
cuándo es necesario el tercer nivel.

**Factores:**
- $A$ = Temperatura de reacción: 60 (−1), 75 (0), 90 °C (+1)
- $B$ = Concentración del catalizador: 10 (−1), 15 (0), 20 g/L (+1)

**Respuesta:** Rendimiento de reacción (%)

**Dataset:** `../../datos/rendimiento-reaccion-3k.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy import stats

df = pd.read_csv('../../datos/rendimiento-reaccion-3k.csv')
print(df)

## 1. Subconjunto $2^2$: solo los 4 vértices

Extraemos las 4 corridas con $x_1, x_2 \in \{-1, +1\}$ y ajustamos un modelo puramente lineal.

In [ ]:
df22 = df[df['x1'].isin([-1, 1]) & df['x2'].isin([-1, 1])].copy()
print('Corridas del subconjunto 2^2:')
print(df22[['x1','x2','rendimiento']])

modelo_lineal = smf.ols('rendimiento ~ x1 + x2 + x1:x2', data=df22).fit()
print('\nModelo lineal (2^2):')
print(modelo_lineal.summary())
print()
print('⚠ NOTA: El 2^2 con interacción tiene 4 parámetros y solo 4 observaciones')
print('   → modelo SATURADO: 0 grados de libertad para el error, R²=1, sin p-valores.')
print('   Esto no es un error del software — es exactamente la limitación que motiva el')
print('   tercer nivel: sin réplicas ni puntos intermedios, no podemos estimar el error.')

## 2. Diseño $3^2$ completo: modelo cuadrático

Con las 9 corridas ajustamos el modelo de segundo orden incluyendo términos cuadráticos.

In [ ]:
modelo_cuad = smf.ols('rendimiento ~ x1 + x2 + I(x1**2) + I(x2**2) + x1:x2',
                      data=df).fit()
print('Modelo cuadrático (3^2):')
print(modelo_cuad.summary())

## 3. Diagnóstico de falta de ajuste

Comparamos los dos modelos: el modelo lineal del $2^2$ genera predicciones para las
corridas del nivel central ($x_1=0, x_2=0$) que podemos contrastar con la observación real.
Una discrepancia grande indica curvatura.

In [ ]:
# Predicción del modelo lineal en el punto central
centro = pd.DataFrame({'x1': [0], 'x2': [0]})
pred_lineal_centro = modelo_lineal.predict(centro)[0]
y_centro_obs = df.loc[(df['x1']==0) & (df['x2']==0), 'rendimiento'].values[0]

print(f'Predicción lineal en (0,0): {pred_lineal_centro:.2f}')
print(f'Observación real en (0,0):  {y_centro_obs:.2f}')
print(f'Diferencia (curvatura):     {y_centro_obs - pred_lineal_centro:.2f}')

# Coeficientes cuadráticos del modelo 3^2
b11 = modelo_cuad.params['I(x1 ** 2)']
b22 = modelo_cuad.params['I(x2 ** 2)']
p11 = modelo_cuad.pvalues['I(x1 ** 2)']
p22 = modelo_cuad.pvalues['I(x2 ** 2)']
print(f'\nβ₁₁ (curvatura A): {b11:.3f}  (p={p11:.4f})')
print(f'β₂₂ (curvatura B): {b22:.3f}  (p={p22:.4f})')

## 4. Tabla de ANOVA comparativa

In [ ]:
anova_cuad = sm.stats.anova_lm(modelo_cuad, typ=1)
print('ANOVA — Modelo cuadrático (3^2):')
print(anova_cuad.round(4))

## 5. Visualización: superficie de respuesta ajustada

In [ ]:
x1g, x2g = np.meshgrid(np.linspace(-1.2, 1.2, 50), np.linspace(-1.2, 1.2, 50))
grid = pd.DataFrame({'x1': x1g.ravel(), 'x2': x2g.ravel()})
zg = modelo_cuad.predict(grid).values.reshape(x1g.shape)

fig = plt.figure(figsize=(12, 5))
ax0 = fig.add_subplot(121)
ax3d = fig.add_subplot(122, projection='3d')

# Contornos
cp = ax0.contourf(x1g, x2g, zg, levels=15, cmap='RdYlGn')
ax0.scatter(df['x1'], df['x2'], c='black', zorder=5)
ax0.set_xlabel('$x_1$ (Temperatura)', fontsize=11)
ax0.set_ylabel('$x_2$ (Concentración)', fontsize=11)
ax0.set_title('Curvas de nivel — Modelo $3^2$')
plt.colorbar(cp, ax=ax0, label='Rendimiento (%)')

# Superficie 3D
ax3d.plot_surface(x1g, x2g, zg, cmap='RdYlGn', alpha=0.8)
ax3d.scatter(df['x1'], df['x2'], df['rendimiento'], c='black', s=40, zorder=5)
ax3d.set_xlabel('$x_1$')
ax3d.set_ylabel('$x_2$')
ax3d.set_zlabel('Rendimiento')
ax3d.set_title('Superficie de respuesta')

plt.tight_layout()
plt.show()

## 6. Conclusión

- El diseño **$2^2$** (4 vértices) ajusta solo relaciones lineales y no puede detectar curvatura.
- El diseño **$3^2$** revela si $\hat\beta_{11}$ o $\hat\beta_{22}$ son significativos.
- Si la curvatura es importante, el $3^2$ indica que un **CCD** o **Box-Behnken** permitirá
  localizar el óptimo con mayor precisión.

> **Regla práctica.** Cuando la diferencia $|y_{\text{centro}} - \hat{y}_{\text{lineal}}|$
> es grande en relación con el error experimental, la curvatura es real y justifica tres niveles.